# LeetCode 2033 — Minimum Operations to Make a Uni-Value Grid bằng Python

Link đề: https://leetcode.com/problems/minimum-operations-to-make-a-uni-value-grid/ · Độ khó: **Medium** · Chủ đề: Array, Math, Sorting, Matrix

Skeleton LeetCode:

```python
class Solution:
    def minOperations(self, grid: List[List[int]], x: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra điều kiện **khả thi**: mọi phần tử phải có **cùng số dư** khi chia cho `x` (vì mỗi thao tác chỉ thay đổi bội của `x`).
2. Đưa bài về **tối thiểu tổng khoảng cách tuyệt đối** `Σ|a_i − t|` và chứng minh **trung vị** là điểm tối ưu.
3. Cài đặt bằng **sắp xếp + trung vị**, bằng **tổng tiền tố** để đánh giá mọi ứng viên trong `O(n)`, và bằng **đếm** (giá trị ≤ 10⁴) để tìm trung vị không cần sắp xếp so sánh.
4. Làm phẳng ma trận bằng list comprehension / `itertools.chain`.
5. Kiểm thử đối chiếu brute force thử mọi đích có thể trên dữ liệu ngẫu nhiên seed cố định.

## 2. Đề bài & ví dụ

Cho lưới số nguyên `grid` cỡ `m × n` và số nguyên `x`. Một thao tác: **cộng** `x` vào hoặc **trừ** `x` khỏi một phần tử bất kỳ. Lưới **đồng giá trị** nếu mọi phần tử bằng nhau.

Trả về số thao tác **ít nhất** để lưới đồng giá trị, hoặc `-1` nếu không thể.

**Ví dụ 1:** `grid = [[2,4],[6,8]], x = 2` → `4` (đưa hết về 4: `2→4` 1 lần, `6→4` 1 lần, `8→4` 2 lần).

**Ví dụ 2:** `grid = [[1,5],[2,3]], x = 1` → `5` (đưa hết về 3).

**Ví dụ 3:** `grid = [[1,2],[3,4]], x = 2` → `-1`.

**Ràng buộc:** `1 ≤ m, n ≤ 10⁵`, `1 ≤ m·n ≤ 10⁵`, `1 ≤ x, grid[i][j] ≤ 10⁴`.

In [1]:
import random
import time
from itertools import chain, accumulate
from typing import List

examples = [([[2, 4], [6, 8]], 2, 4), ([[1, 5], [2, 3]], 1, 5), ([[1, 2], [3, 4]], 2, -1)]
for g, x, exp in examples:
    print(g, "x =", x, "-> kỳ vọng", exp)

[[2, 4], [6, 8]] x = 2 -> kỳ vọng 4
[[1, 5], [2, 3]] x = 1 -> kỳ vọng 5
[[1, 2], [3, 4]] x = 2 -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1 Làm phẳng ma trận

Bài không quan tâm vị trí ô, nên đổi lưới thành một danh sách `a`. Hai cách phổ biến: list comprehension lồng nhau hoặc `itertools.chain.from_iterable`.

In [2]:
g = [[2, 4], [6, 8]]
print([v for row in g for v in row], list(chain.from_iterable(g)))

[2, 4, 6, 8] [2, 4, 6, 8]


### 3.2 Bất biến số dư

Cộng/trừ `x` không đổi `a mod x`. Muốn hai phần tử bằng nhau thì chúng phải cùng số dư ngay từ đầu. Ngược lại nếu mọi phần tử cùng dư `r`, thì mọi đích `t ≡ r (mod x)` đều đạt được, với số thao tác cho phần tử `a` là `|a − t| / x`.

In [3]:
for g, x, _ in examples:
    a = [v for row in g for v in row]
    print(a, "x =", x, "số dư:", [v % x for v in a], "-> cùng dư?", len({v % x for v in a}) == 1)

[2, 4, 6, 8] x = 2 số dư: [0, 0, 0, 0] -> cùng dư? True
[1, 5, 2, 3] x = 1 số dư: [0, 0, 0, 0] -> cùng dư? True
[1, 2, 3, 4] x = 2 số dư: [1, 0, 1, 0] -> cùng dư? False


### 3.3 Trung vị và tổng khoảng cách tuyệt đối

Với dãy đã sắp `b[0] ≤ ... ≤ b[n−1]`, hàm `f(t) = Σ|b_i − t|` đạt nhỏ nhất tại **trung vị** `b[n // 2]` (với `n` chẵn, mọi `t` trong `[b[n/2 − 1], b[n/2]]` đều tối ưu). Trung bình cộng **không** tối ưu cho tổng trị tuyệt đối (nó tối ưu cho tổng bình phương).

In [4]:
b = sorted([1, 2, 3, 5, 100])
f = lambda t: sum(abs(v - t) for v in b)
print("trung vị", b[len(b) // 2], "-> f =", f(b[len(b) // 2]), "| trung bình", sum(b) / len(b), "-> f =", f(sum(b) / len(b)))
print("f(t) với t = 0..6:", [f(t) for t in range(7)])

trung vị 3 -> f = 102 | trung bình 22.2 -> f = 155.6
f(t) với t = 0..6: [111, 106, 103, 102, 103, 104, 107]


### 3.4 Tổng tiền tố trên mảng đã sắp

`S[k] = b[0] + ... + b[k−1]`. Với đích `t = b[k]`: các phần tử trái (`k` phần tử) cần `k·t − S[k]`, các phần tử phải cần `(S[n] − S[k+1]) − (n − k − 1)·t`. Tính `f(b[k])` trong `O(1)` cho mọi `k`.

In [5]:
b = [1, 2, 3, 5, 100]; n = len(b); S = [0] + list(accumulate(b))
k = 2; t = b[k]
cost = (k * t - S[k]) + (S[n] - S[k + 1] - (n - k - 1) * t)
print("f(b[2]) qua tiền tố =", cost, "| trực tiếp =", sum(abs(v - t) for v in b))

f(b[2]) qua tiền tố = 102 | trực tiếp = 102


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: `a = [1, 5, 2, 3]`, `x = 1`. Thử các đích:

| t | |1−t|+|5−t|+|2−t|+|3−t| |
|---|---|
| 1 | 0+4+1+2 = 7 |
| 2 | 1+3+0+1 = 5 |
| 3 | 2+2+1+0 = 5 |
| 4 | 3+1+2+1 = 7 |

Sắp xếp `[1, 2, 3, 5]`: trung vị nằm giữa 2 và 3 ⇒ cả `t = 2` và `t = 3` đều cho `5` ✓.

Trực giác "kéo co": nếu có nhiều phần tử ở bên phải `t` hơn bên trái, dịch `t` sang phải một đơn vị làm giảm tổng (bên phải giảm nhiều hơn bên trái tăng). Cân bằng khi hai bên có số phần tử bằng nhau — đó là trung vị.

In [6]:
a = [1, 5, 2, 3]
for t in range(0, 7):
    left = sum(1 for v in a if v < t); right = sum(1 for v in a if v > t)
    print(f"t={t}: tổng = {sum(abs(v - t) for v in a)}, số phần tử < t: {left}, > t: {right}")

t=0: tổng = 11, số phần tử < t: 0, > t: 4
t=1: tổng = 7, số phần tử < t: 0, > t: 3
t=2: tổng = 5, số phần tử < t: 1, > t: 2
t=3: tổng = 5, số phần tử < t: 2, > t: 1
t=4: tổng = 7, số phần tử < t: 3, > t: 1
t=5: tổng = 9, số phần tử < t: 3, > t: 0
t=6: tổng = 13, số phần tử < t: 4, > t: 0


## 5. Cách 1 — Brute Force / baseline: thử mọi đích có thể

**Ý tưởng:** kiểm tra cùng số dư; rồi thử mọi `t` từ `min(a)` tới `max(a)` bước `x` (các đích hợp lệ nằm trong khoảng này — đi ra ngoài chỉ tốn thêm), tính tổng `|a_i − t| / x`.

In [7]:
def min_ops_brute(grid, x):
    a = [v for row in grid for v in row]
    if len({v % x for v in a}) > 1:
        return -1
    lo, hi = min(a), max(a)
    best = None
    for t in range(lo, hi + 1, x):
        cost = sum(abs(v - t) // x for v in a)
        if best is None or cost < best:
            best = cost
    return best

for g, x, exp in examples:
    got = min_ops_brute(g, x); print(got, "kỳ vọng", exp); assert got == exp

4 kỳ vọng 4
5 kỳ vọng 5
-1 kỳ vọng -1


**Độ phức tạp:** `O(N · (max − min)/x)` với `N = m·n`. Worst case `x = 1`, giá trị trải từ 1 tới `10⁴` ⇒ `10⁵ · 10⁴ = 10⁹` — quá chậm. Space `O(N)`.

**Nút thắt:** ta thử rất nhiều đích, trong khi lý thuyết cho biết đích tối ưu là **trung vị** — chỉ cần một ứng viên.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (khả thi):** **nếu** tồn tại hai phần tử khác số dư mod `x` **thì** trả `-1`.

**Insight 2 (đổi đơn vị):** khi cùng dư `r`, đặt `b_i = (a_i − r) / x` (số nguyên). Số thao tác cho đích `t = r + x·u` là `Σ|b_i − u|` — bài toán **tổng khoảng cách tuyệt đối** trên các số nguyên `b_i`. Thực tế không cần trừ `r`: `|a_i − t| / x` với `t` là một phần tử của mảng (cùng dư) luôn là số nguyên.

**Insight 3 (trung vị):** đích tối ưu là **trung vị** của `a` (một phần tử của mảng ⇒ tự động cùng dư).

**Quy tắc "nếu… thì…":**
- **Nếu** dịch `t` sang phải mà số phần tử `> t` nhiều hơn số phần tử `≤ t` **thì** tổng giảm ⇒ tiếp tục dịch.
- Dừng khi hai bên cân bằng ⇒ trung vị.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Thử mỗi phần tử làm đích (`O(N²)`)

Một bước trung gian: đích tối ưu luôn có thể chọn là **một phần tử** của mảng (hàm `f` tuyến tính từng khúc, cực tiểu tại điểm gãy). Thử cả `N` phần tử.

In [8]:
def min_ops_each_element(grid, x):
    a = [v for row in grid for v in row]
    if len({v % x for v in a}) > 1:
        return -1
    return min(sum(abs(v - t) for v in a) // x for t in set(a))

for g, x, exp in examples:
    assert min_ops_each_element(g, x) == exp
print([min_ops_each_element(g, x) for g, x, _ in examples])

[4, 5, -1]


**Độ phức tạp Cách 2:** `O(K · N)` với `K` = số giá trị khác nhau (≤ `min(N, 10⁴)`) — vẫn tới `10⁹` worst case.

### 7.2 Cách 3 — Sắp xếp + trung vị (bản nộp chính)

**Giải thích từng đoạn:**
1. Làm phẳng thành `a`, kiểm cùng số dư (so với `a[0] % x`).
2. `a.sort()`, `t = a[len(a) // 2]`.
3. Trả `Σ |v − t| // x`.

In [9]:
def min_ops_median(grid, x):
    a = [v for row in grid for v in row]
    r = a[0] % x
    if any(v % x != r for v in a):
        return -1
    a.sort()
    t = a[len(a) // 2]
    return sum(abs(v - t) for v in a) // x

for g, x, exp in examples:
    assert min_ops_median(g, x) == exp
print([min_ops_median(g, x) for g, x, _ in examples])

[4, 5, -1]


**Độ phức tạp Cách 3:** `O(N log N)` do sắp xếp, Space `O(N)`.

### 7.3 Cách 4 — Sắp xếp + tổng tiền tố, đánh giá **mọi** ứng viên trong `O(N)`

Hữu ích khi đề đổi hàm chi phí (không còn là trung vị đơn giản) nhưng vẫn tuyến tính từng khúc: tính `f(b[k])` cho mọi `k` bằng tiền tố và lấy min.

In [10]:
def min_ops_prefix(grid, x):
    a = sorted(v for row in grid for v in row)
    if any((v - a[0]) % x for v in a):
        return -1
    n = len(a); S = [0] + list(accumulate(a))
    best = None
    for k in range(n):
        t = a[k]
        cost = (k * t - S[k]) + (S[n] - S[k + 1] - (n - k - 1) * t)
        if best is None or cost < best:
            best = cost
    return best // x

for g, x, exp in examples:
    assert min_ops_prefix(g, x) == exp
print([min_ops_prefix(g, x) for g, x, _ in examples])

[4, 5, -1]


**Độ phức tạp Cách 4:** `O(N log N)` (sắp) + `O(N)` (quét). Space `O(N)`.

### 7.4 Cách 5 — Đếm (counting) để tìm trung vị trong `O(N + V)`

Vì `1 ≤ grid[i][j] ≤ 10⁴`, đếm tần suất vào mảng `cnt[0..V]`, rồi đi tích lũy tới khi vượt `N // 2` ⇒ đó là trung vị `a[N // 2]` của mảng đã sắp — không cần so sánh để sắp.

In [11]:
def min_ops_counting(grid, x):
    V = 10**4
    cnt = [0] * (V + 1)
    r = grid[0][0] % x; N = 0
    for row in grid:
        for v in row:
            if v % x != r:
                return -1
            cnt[v] += 1; N += 1
    need = N // 2; acc = 0; t = 0
    for v in range(V + 1):
        acc += cnt[v]
        if acc > need:                    # a_sorted[need] = v
            t = v; break
    return sum(c * abs(v - t) for v, c in enumerate(cnt) if c) // x

for g, x, exp in examples:
    assert min_ops_counting(g, x) == exp
print([min_ops_counting(g, x) for g, x, _ in examples])

[4, 5, -1]


In [12]:
class SolutionCounting:
    def minOperations(self, grid: List[List[int]], x: int) -> int:
        V = 10**4
        cnt = [0] * (V + 1)
        r = grid[0][0] % x; N = 0
        for row in grid:
            for v in row:
                if v % x != r:
                    return -1
                cnt[v] += 1
            N += len(row)
        acc, t = 0, 0
        for v in range(V + 1):
            acc += cnt[v]
            if acc > N // 2:
                t = v; break
        return sum(c * abs(v - t) for v, c in enumerate(cnt) if c) // x

print([SolutionCounting().minOperations(g, x) for g, x, _ in examples])

[4, 5, -1]


**Độ phức tạp Cách 5:** `O(N + V)` với `V = 10⁴`. Space `O(V)`.

### 7.5 Cách 6 — Quickselect: tìm phần tử thứ `N//2` trong `O(N)` kỳ vọng

Không cần sắp **toàn bộ** mảng — chỉ cần phần tử đứng ở vị trí `N//2` sau khi sắp. **Quickselect** chọn chốt ngẫu nhiên, chia mảng thành ba phần `< chốt`, `= chốt`, `> chốt`, rồi chỉ đi tiếp vào **một** phần chứa vị trí cần tìm. Kỳ vọng `O(N)` (tổng `N + N/2 + N/4 + ...`), worst case `O(N²)` nhưng cực hiếm với chốt ngẫu nhiên. Viết dạng vòng lặp để không đệ quy sâu.

In [13]:
def quickselect(a, k, rng=random.Random(0)):
    # trả về phần tử nhỏ thứ k (0-indexed) của a
    a = list(a)
    while True:
        pivot = a[rng.randrange(len(a))]
        lo = [v for v in a if v < pivot]
        eq = [v for v in a if v == pivot]
        if k < len(lo):
            a = lo
        elif k < len(lo) + len(eq):
            return pivot
        else:
            k -= len(lo) + len(eq)
            a = [v for v in a if v > pivot]

def min_ops_quickselect(grid, x):
    a = [v for row in grid for v in row]
    if any(v % x != a[0] % x for v in a):
        return -1
    t = quickselect(a, len(a) // 2)
    return sum(abs(v - t) for v in a) // x

for g, x, exp in examples:
    assert min_ops_quickselect(g, x) == exp
rng_ = random.Random(1)
for _ in range(200):
    arr = [rng_.randint(1, 50) for _ in range(rng_.randint(1, 30))]
    assert quickselect(arr, len(arr) // 2) == sorted(arr)[len(arr) // 2]
print([min_ops_quickselect(g, x) for g, x, _ in examples], "| quickselect khớp sorted()[N//2] trên 200 mảng")

[4, 5, -1] | quickselect khớp sorted()[N//2] trên 200 mảng


**Độ phức tạp Cách 6:** `O(N)` kỳ vọng, `O(N)` bộ nhớ (do tạo list con). Trong Python thực tế, `sort()` viết bằng C thường **nhanh hơn** quickselect viết bằng Python thuần với `N = 10⁵` — bài học: độ phức tạp tiệm cận không phải là tất cả; hằng số cũng quan trọng.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 `[[2,4],[6,8]], x = 2`:

| Bước | Giá trị |
|---|---|
| làm phẳng | `[2, 4, 6, 8]` |
| số dư mod 2 | `[0, 0, 0, 0]` ⇒ cùng dư ✓ |
| sắp xếp | `[2, 4, 6, 8]` |
| trung vị `a[4 // 2]` | `6` |
| số thao tác | `(4 + 2 + 0 + 2) / 2 = 4` |

Chọn `t = 6` hay `t = 4` (đề minh họa 4) đều cho 4 — với `N` chẵn có cả một khoảng trung vị tối ưu.

In [14]:
def trace_median(grid, x):
    a = [v for row in grid for v in row]
    print("làm phẳng:", a, "| số dư:", [v % x for v in a])
    if len({v % x for v in a}) > 1:
        print("-> khác số dư, trả -1"); return -1
    a.sort(); t = a[len(a) // 2]
    print("sắp xếp:", a, "| trung vị a[{}] = {}".format(len(a) // 2, t))
    total = 0
    for v in a:
        ops = abs(v - t) // x; total += ops
        print(f"  {v} -> {t}: |{v}-{t}|/{x} = {ops} thao tác, cộng dồn {total}")
    return total

print("Đáp án:", trace_median(*examples[0][:2]))
print()
print("Đáp án:", trace_median(*examples[1][:2]))
print()
print("Đáp án:", trace_median(*examples[2][:2]))

làm phẳng: [2, 4, 6, 8] | số dư: [0, 0, 0, 0]
sắp xếp: [2, 4, 6, 8] | trung vị a[2] = 6
  2 -> 6: |2-6|/2 = 2 thao tác, cộng dồn 2
  4 -> 6: |4-6|/2 = 1 thao tác, cộng dồn 3
  6 -> 6: |6-6|/2 = 0 thao tác, cộng dồn 3
  8 -> 6: |8-6|/2 = 1 thao tác, cộng dồn 4
Đáp án: 4

làm phẳng: [1, 5, 2, 3] | số dư: [0, 0, 0, 0]
sắp xếp: [1, 2, 3, 5] | trung vị a[2] = 3
  1 -> 3: |1-3|/1 = 2 thao tác, cộng dồn 2
  2 -> 3: |2-3|/1 = 1 thao tác, cộng dồn 3
  3 -> 3: |3-3|/1 = 0 thao tác, cộng dồn 3
  5 -> 3: |5-3|/1 = 2 thao tác, cộng dồn 5
Đáp án: 5

làm phẳng: [1, 2, 3, 4] | số dư: [1, 0, 1, 0]
-> khác số dư, trả -1
Đáp án: -1


In [15]:
def trace_prefix(grid, x):
    a = sorted(v for row in grid for v in row); n = len(a); S = [0] + list(accumulate(a))
    for k in range(n):
        t = a[k]; L = k * t - S[k]; R = S[n] - S[k + 1] - (n - k - 1) * t
        print(f"k={k}, t={t}: bên trái cần {L}, bên phải cần {R} -> tổng {L + R} (÷{x} = {(L + R) // x})")

trace_prefix([[1, 5], [2, 3]], 1)

k=0, t=1: bên trái cần 0, bên phải cần 7 -> tổng 7 (÷1 = 7)
k=1, t=2: bên trái cần 1, bên phải cần 4 -> tổng 5 (÷1 = 5)
k=2, t=3: bên trái cần 3, bên phải cần 2 -> tổng 5 (÷1 = 5)
k=3, t=5: bên trái cần 9, bên phải cần 0 -> tổng 9 (÷1 = 9)


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def minOperations(self, grid: List[List[int]], x: int) -> int:
        a = [v for row in grid for v in row]
        r = a[0] % x
        for v in a:
            if v % x != r:               # thao tác không đổi số dư mod x
                return -1
        a.sort()
        t = a[len(a) // 2]               # trung vị tối thiểu tổng |a_i - t|
        return sum(abs(v - t) for v in a) // x

for g, x, exp in examples:
    assert Solution().minOperations(g, x) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Khả thi.** Mỗi thao tác đổi một phần tử đi `±x` ⇒ `a mod x` bất biến. Lưới đồng giá trị ⇒ mọi phần tử cùng dư lúc kết thúc ⇒ cùng dư lúc ban đầu. Ngược lại, nếu cùng dư `r`, đích `t ≡ r` bất kỳ đạt được với `|a_i − t|/x` thao tác cho mỗi phần tử (và không thể ít hơn vì mỗi thao tác đổi đúng `x`).

**Trung vị tối ưu.** Với `b` đã sắp, xét `f(t) = Σ|b_i − t|`. Khi tăng `t` thêm `δ > 0` nhỏ (không vượt phần tử nào), `f` thay đổi `δ·(L − R)` với `L = #{b_i ≤ t}`, `R = #{b_i > t}`. Tại `t = b[n//2]`, `L ≥ n//2 + 1 > R` khi đi sang phải (tăng), và khi đi sang trái số phần tử `≥ t` là `≥ n − n//2 ≥ n//2` ≥ số phần tử `< t` ⇒ không giảm. Vậy `f` đạt min tại trung vị. Vì trung vị là một phần tử của mảng nên nó cùng dư `r` — hợp lệ.

Kiểm chứng thực nghiệm dưới đây: với mảng ngẫu nhiên (cùng dư), `f(trung vị)` luôn bằng min trên mọi đích có thể.

In [17]:
rng = random.Random(10)
for _ in range(500):
    x = rng.randint(1, 5); r = rng.randint(0, x - 1)
    a = [r + x * rng.randint(0, 20) for _ in range(rng.randint(1, 15))]
    a = [v if v >= 1 else v + x for v in a]
    a.sort(); t = a[len(a) // 2]
    best = min(sum(abs(v - u) for v in a) for u in range(min(a), max(a) + 1, x))
    assert sum(abs(v - t) for v in a) == best
print("Trung vị tối ưu trên 500 mảng ngẫu nhiên")

Trung vị tối ưu trên 500 mảng ngẫu nhiên


## 11. Phân tích Time / Space

Gọi `N = m·n`, `V = 10⁴` (giá trị lớn nhất).

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi đích | `O(N · V / x)` | `O(N)` |
| 2. Mỗi phần tử làm đích | `O(N · K)` | `O(N)` |
| 3. Sắp + trung vị | `O(N log N)` | `O(N)` |
| 4. Sắp + tiền tố | `O(N log N)` | `O(N)` |
| 5. Đếm | `O(N + V)` | `O(V)` |
| 6. Quickselect | `O(N)` kỳ vọng, `O(N²)` worst | `O(N)` |

In [18]:
rng = random.Random(2033)
x = 3
big = [[3 * rng.randint(1, 3333) for _ in range(1000)] for _ in range(100)]       # 10^5 ô, cùng dư
for name, fn in [("median", min_ops_median), ("prefix", min_ops_prefix), ("counting", min_ops_counting),
                 ("quickselect", min_ops_quickselect), ("Solution", lambda g, x: Solution().minOperations(g, x))]:
    t = time.perf_counter(); r = fn(big, x)
    print(f"{name:9s}: {r} ({time.perf_counter() - t:.3f}s)")

median   : 83461571 (0.020s)
prefix   : 83461571 (0.035s)
counting : 83461571 (0.005s)


quickselect: 83461571 (0.032s)


Solution : 83461571 (0.024s)


## 12. Bộ test

Ví dụ đề; biên (một ô, mọi ô bằng nhau, `x` lớn hơn mọi hiệu, lưới 1 hàng / 1 cột, `N` chẵn với hai trung vị, một ô khác dư duy nhất); đối chiếu brute force trên dữ liệu ngẫu nhiên (một nửa bị "phá" số dư).

In [19]:
fns = {
    "brute": min_ops_brute, "each": min_ops_each_element, "median": min_ops_median,
    "prefix": min_ops_prefix, "counting": min_ops_counting, "quickselect": min_ops_quickselect,
    "Solution": lambda g, x: Solution().minOperations(g, x),
    "SolutionCounting": lambda g, x: SolutionCounting().minOperations(g, x),
}
edge = [
    ([[7]], 3, 0), ([[5, 5], [5, 5]], 2, 0), ([[1, 10000]], 9999, 1),
    ([[1, 2, 3, 4, 5]], 1, 6), ([[1], [3], [5], [7]], 2, 4),
    ([[2, 4, 6, 9]], 2, -1), ([[10000, 1]], 1, 9999),
]
for g, x, exp in [(g, x, e) for g, x, e in examples] + edge:
    for name, fn in fns.items():
        assert fn(g, x) == exp, (name, g, x, fn(g, x))
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [20]:
rng = random.Random(42)
neg = 0
for _ in range(500):
    m, n = rng.randint(1, 4), rng.randint(1, 4); x = rng.randint(1, 6); r = rng.randint(0, x - 1)
    g = [[r + x * rng.randint(0, 15) or x for _ in range(n)] for _ in range(m)]
    if rng.random() < 0.4:
        i, j = rng.randrange(m), rng.randrange(n); g[i][j] = rng.randint(1, 100)
    want = min_ops_brute(g, x); neg += want == -1
    for name, fn in fns.items():
        assert fn(g, x) == want, (name, g, x)
print(f"500 test ngẫu nhiên ({neg} ca -1): mọi cách khớp brute force")

500 test ngẫu nhiên (119 ca -1): mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra số dư (trả về số thao tác dù không thể)

In [21]:
def wrong_no_mod_check(grid, x):
    a = sorted(v for row in grid for v in row)            # SAI: không kiểm v % x
    t = a[len(a) // 2]
    return sum(abs(v - t) for v in a) // x

g, x, exp = examples[2]
print("sai:", wrong_no_mod_check(g, x), "| đúng:", exp)

sai: 2 | đúng: -1


In [22]:
print("sửa:", Solution().minOperations(g, x))

sửa: -1


### Lỗi 2 — Dùng **trung bình cộng** thay cho trung vị

Trung bình bị kéo bởi giá trị ngoại lai; còn có thể không cùng số dư / không nguyên.

In [23]:
def wrong_mean(grid, x):
    a = [v for row in grid for v in row]
    if len({v % x for v in a}) > 1: return -1
    mean = sum(a) / len(a)
    r = a[0] % x
    t = r + x * round((mean - r) / x)                     # SAI: đích gần trung bình nhất (cùng dư)
    return sum(abs(v - t) for v in a) // x

g, x = [[1, 1, 1, 100]], 1
print("sai:", wrong_mean(g, x), "| đúng:", Solution().minOperations(g, x))

sai: 149 | đúng: 99


In [24]:
print("sửa (trung vị):", min_ops_median(g, x))

sửa (trung vị): 99


### Lỗi 3 — Quên chia cho `x`

`Σ|a_i − t|` là tổng **khoảng cách**, còn số thao tác là khoảng cách chia `x`.

In [25]:
def wrong_no_divide(grid, x):
    a = sorted(v for row in grid for v in row)
    if len({v % x for v in a}) > 1: return -1
    t = a[len(a) // 2]
    return sum(abs(v - t) for v in a)                     # SAI: thiếu // x

g, x, exp = examples[0]
print("sai:", wrong_no_divide(g, x), "| đúng:", exp)

sai: 8 | đúng: 4


In [26]:
print("sửa:", min_ops_median(g, x))

sửa: 4


### Lỗi 4 — Lấy "trung vị" bằng trung bình hai phần tử giữa khi `N` chẵn

`(a[n/2 − 1] + a[n/2]) // 2` có thể **không cùng số dư** với các phần tử: khi đó `|v − t|` không chia hết cho `x`, phép `// x` cắt bớt phần lẻ và cho một con số **nhỏ hơn thực tế** — trông như "tốt hơn" nhưng không đạt được.

In [27]:
def wrong_even_median(grid, x):
    a = sorted(v for row in grid for v in row)
    if len({v % x for v in a}) > 1: return -1
    n = len(a)
    t = a[n // 2] if n % 2 else (a[n // 2 - 1] + a[n // 2]) // 2  # SAI khi n chẵn: t có thể khác số dư
    return sum(abs(v - t) // x for v in a)

g, x = [[2, 4], [6, 12]], 2
print("sai:", wrong_even_median(g, x), "(đích 5 lẻ, không đạt được) | đúng:", Solution().minOperations(g, x))

sai: 4 (đích 5 lẻ, không đạt được) | đúng: 6


In [28]:
print("sửa (chọn một phần tử giữa, ví dụ a[n//2]):", min_ops_median(g, x))

sửa (chọn một phần tử giữa, ví dụ a[n//2]): 6


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đưa mọi phần tử về một giá trị, chi phí ∝ |chênh lệch|"** ⇒ **trung vị**. Chi phí ∝ bình phương ⇒ trung bình.
- **Thao tác ±x** ⇒ kiểm bất biến **mod x** trước tiên.
- Lưới/ma trận mà vị trí không quan trọng ⇒ **làm phẳng**.
- Giá trị nhỏ (`≤ 10⁴`) ⇒ có thể **đếm** thay vì sắp xếp.
- Cần đánh giá nhiều đích ⇒ **tổng tiền tố** trên mảng đã sắp.

## 15. Biến thể của bài

1. **LeetCode 462 — Minimum Moves to Equal Array Elements II:** chính là bài này với `x = 1`.
2. **LeetCode 296 — Best Meeting Point (Hard, Premium):** trung vị riêng theo hàng và theo cột.
3. **LeetCode 2448 — Minimum Cost to Make Array Equal (Hard):** mỗi phần tử có trọng số chi phí ⇒ **trung vị có trọng số**.
4. **Chi phí cộng khác chi phí trừ** (bài tập 5): đích tối ưu là một **phân vị** (quantile) khác trung vị.
5. **Chỉ được cộng** (không được trừ): đích phải là `max` ⇒ tổng `(max − a_i)/x`.

Minh họa biến thể 3 (trung vị có trọng số, tự dựng, kiểm bằng brute force):

In [29]:
def weighted_median_cost(nums, cost):
    pairs = sorted(zip(nums, cost)); total = sum(cost); acc = 0
    for v, c in pairs:
        acc += c
        if 2 * acc >= total:
            t = v; break
    return sum(abs(v - t) * c for v, c in zip(nums, cost))

rng = random.Random(2448)
for _ in range(400):
    k = rng.randint(1, 8)
    nums = [rng.randint(1, 20) for _ in range(k)]; cost = [rng.randint(1, 9) for _ in range(k)]
    brute = min(sum(abs(v - t) * c for v, c in zip(nums, cost)) for t in range(1, 21))
    assert weighted_median_cost(nums, cost) == brute
print("Trung vị có trọng số khớp brute force; ví dụ:", weighted_median_cost([1, 3, 5, 2], [2, 3, 1, 14]))

Trung vị có trọng số khớp brute force; ví dụ: 8


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi đích | duyệt `t` bước `x` | `O(N·V/x)` | `O(N)` | kiểm tra |
| 2. Mỗi phần tử làm đích | đích là một phần tử | `O(N·K)` | `O(N)` | bước trung gian |
| 3. Sắp + trung vị | `a[N//2]` | `O(N log N)` | `O(N)` | **nộp bài** |
| 4. Sắp + tiền tố | đánh giá mọi `k` | `O(N log N)` | `O(N)` | hàm chi phí phức tạp hơn |
| 5. Đếm | trung vị qua tích lũy | `O(N + V)` | `O(V)` | giá trị nhỏ, cần nhanh |
| 6. Quickselect | chọn phần tử thứ `N//2` | `O(N)` kỳ vọng | `O(N)` | giá trị lớn, không muốn sắp toàn bộ |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `minOperations([[1,4,7],[10,13,16]], 3)`.
2. **Tính tay:** `minOperations([[5,1],[9,13],[17,21]], 4)` — lập bảng trace.
3. **Điền TODO:** `same_residue(grid, x)` trả `True` nếu mọi phần tử cùng dư mod `x`.
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** `min_cost_asym(nums, add_cost, sub_cost)` — mỗi lần **tăng 1** tốn `add_cost`, mỗi lần **giảm 1** tốn `sub_cost`; tìm chi phí nhỏ nhất để mọi phần tử bằng nhau. Gợi ý: đích là phân vị thứ `add_cost/(add_cost+sub_cost)`. Kiểm bằng brute force.

In [30]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def same_residue(grid, x):
    # TODO
    pass

In [31]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_min_operations(grid, x):
    # TODO
    pass

In [32]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def min_cost_asym(nums, add_cost, sub_cost):
    # TODO: thử các phần tử đã sắp làm đích bằng tiền tố, hoặc tìm phân vị
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Mọi phần tử dư 1 mod 3. (2) Sắp `[1,4,7,10,13,16]`, trung vị `a[3] = 10`. (3) Tổng khoảng cách `9+6+3+0+3+6 = 27` ⇒ `27 / 3 = 9`.

**Bài 2.** (1) Mọi phần tử dư 1 mod 4. (2) Sắp `[1,5,9,13,17,21]`, trung vị `13`. (3) `(12+8+4+0+4+8)/4 = 9`.

**Bài 3.** (1) `r = grid[0][0] % x`. (2) Duyệt mọi ô. (3) `all(v % x == r ...)`.

**Bài 4.** (1) Làm phẳng + kiểm dư. (2) Sắp, lấy `a[N//2]`. (3) `Σ|v − t| // x`.

**Bài 5.** (1) Chi phí đích `t`: `add·Σ_{v<t}(t − v) + sub·Σ_{v>t}(v − t)` — tuyến tính từng khúc, cực tiểu tại một phần tử. (2) Dùng tiền tố như Cách 4 để đánh giá mọi phần tử. (3) Lấy min.

## 19. Đáp án tham khảo

In [33]:
# Bài 1, 2
assert Solution().minOperations([[1, 4, 7], [10, 13, 16]], 3) == 9
assert Solution().minOperations([[5, 1], [9, 13], [17, 21]], 4) == 9
print("Bài 1: 9 | Bài 2: 9")
trace_median([[5, 1], [9, 13], [17, 21]], 4)

Bài 1: 9 | Bài 2: 9
làm phẳng: [5, 1, 9, 13, 17, 21] | số dư: [1, 1, 1, 1, 1, 1]
sắp xếp: [1, 5, 9, 13, 17, 21] | trung vị a[3] = 13
  1 -> 13: |1-13|/4 = 3 thao tác, cộng dồn 3
  5 -> 13: |5-13|/4 = 2 thao tác, cộng dồn 5
  9 -> 13: |9-13|/4 = 1 thao tác, cộng dồn 6
  13 -> 13: |13-13|/4 = 0 thao tác, cộng dồn 6
  17 -> 13: |17-13|/4 = 1 thao tác, cộng dồn 7
  21 -> 13: |21-13|/4 = 2 thao tác, cộng dồn 9


9

In [34]:
# Bài 3, 4
def same_residue_ans(grid, x):
    r = grid[0][0] % x
    return all(v % x == r for row in grid for v in row)

my_min_operations_ans = min_ops_median
rng = random.Random(3)
for _ in range(300):
    x = rng.randint(1, 5); g = [[rng.randint(1, 30) for _ in range(rng.randint(1, 3))] for _ in range(rng.randint(1, 3))]
    assert same_residue_ans(g, x) == (min_ops_brute(g, x) != -1)
    assert my_min_operations_ans(g, x) == min_ops_brute(g, x)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [35]:
# Bài 5
def min_cost_asym_ans(nums, add_cost, sub_cost):
    a = sorted(nums); n = len(a); S = [0] + list(accumulate(a)); best = None
    for k in range(n):
        t = a[k]
        up = k * t - S[k]                               # các phần tử bên trái phải TĂNG lên t
        down = S[n] - S[k + 1] - (n - k - 1) * t        # các phần tử bên phải phải GIẢM xuống t
        c = add_cost * up + sub_cost * down
        best = c if best is None or c < best else best
    return best

def min_cost_asym_brute(nums, add_cost, sub_cost):
    return min(sum(add_cost * (t - v) if v < t else sub_cost * (v - t) for v in nums)
               for t in range(min(nums), max(nums) + 1))

rng = random.Random(5)
for _ in range(400):
    nums = [rng.randint(1, 25) for _ in range(rng.randint(1, 9))]
    A, B = rng.randint(1, 6), rng.randint(1, 6)
    assert min_cost_asym_ans(nums, A, B) == min_cost_asym_brute(nums, A, B)
print("Bài 5: OK; [1, 2, 10], tăng 1đ, giảm 5đ ->", min_cost_asym_ans([1, 2, 10], 1, 5))

Bài 5: OK; [1, 2, 10], tăng 1đ, giảm 5đ -> 17


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đưa về cùng giá trị với bước ±x":**

```python
a = [v for row in grid for v in row]
if any(v % x != a[0] % x for v in a): return -1
a.sort(); t = a[len(a) // 2]
return sum(abs(v - t) for v in a) // x
```

In [36]:
def template(grid, x):
    a = [v for row in grid for v in row]
    if any(v % x != a[0] % x for v in a): return -1
    a.sort(); t = a[len(a) // 2]
    return sum(abs(v - t) for v in a) // x
print([template(g, x) for g, x, _ in examples])

[4, 5, -1]


**Checklist:**

- [ ] Kiểm **cùng số dư** mod `x` ⇒ khác thì `-1`.
- [ ] Đích là **trung vị** (một phần tử của mảng đã sắp), không phải trung bình.
- [ ] Với `N` chẵn, chọn `a[N//2]` (hoặc `a[N//2 − 1]`), không lấy trung bình hai số giữa.
- [ ] Chia tổng khoảng cách cho `x`.
- [ ] Làm phẳng đúng cả lưới (mọi hàng).
- [ ] Đã thử: một ô, mọi ô bằng nhau, một ô khác dư.